# Phase 4 — Taxonomy & Conformity Validation

**Weeks 10–12 · Goal:** Validate LLM answers against a classification ontology (allowed vs forbidden labels).

## What you will learn

- Loading an **RDF/Turtle** taxonomy (`data/taxonomies/classification.ttl`)
- **Entity extraction** — find classification terms in queries and answers
- **Fuzzy linking** — map typos to canonical labels with rapidfuzz
- **Conformity scoring** — flag forbidden labels (e.g. `SECRET-TOP-SECRET`)
- Pipeline hook — every `RAGPipeline.query()` can attach conformity metadata

> Used in production API responses as `metadata.conformity`.


In [1]:
import sys
from pathlib import Path

# Run from notebooks/ — project root is one level up
ROOT = Path.cwd()
if not (ROOT / "shared").exists() and (ROOT.parent / "shared").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT.resolve()}")


Project root: C:\Users\dyh\Desktop\RAG


In [2]:
from phases.phase_04_taxonomy_generation.ontology.loader import load_taxonomy

ttl = ROOT / "data/taxonomies/classification.ttl"
tax = load_taxonomy(ttl)
print("Allowed labels:", tax.allowed_labels[:5], "...")
print("Forbidden labels:", tax.forbidden_labels[:5], "...")


ImportError: Install rdflib: pip install -e '.[phase4]'

In [ ]:
from phases.phase_04_taxonomy_generation.linking.entity_extractor import extract_classification_terms

text = "Please classify this document as SECRET-TOP-SECRET for internal review."
terms = extract_classification_terms(text)
print("Extracted terms:", terms)


In [ ]:
from phases.phase_04_taxonomy_generation.validation.conformity_validator import ConformityValidator

validator = ConformityValidator()
for query, answer in [
    ("Classify as INTERNAL", "This document is INTERNAL."),
    ("Classify as SECRET-TOP-SECRET", "The classification is SECRET-TOP-SECRET."),
    ("Classify as SECRET-TOP-SECRET", "I cannot classify this as SECRET-TOP-SECRET; allowed labels are PUBLIC, INTERNAL, CONFIDENTIAL."),
]:
    result = validator.validate(query=query, answer=answer)
    print(f"Q: {query[:40]}...")
    print(f"   score={result.score:.2f} flagged={result.flagged} reason={result.reason[:60] if result.reason else '-'}")
    print()


## Block forbidden answers

Set `block_forbidden_classifications=True` in `PipelineConfig` to replace non-compliant answers with a refusal message.

**Next:** [05_evaluation.ipynb](05_evaluation.ipynb)
